# RQ2: Task Type Sanity Check

## Imports & Load Dataset

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import cohen_kappa_score

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

filtered_df = pd.read_csv('data/filtered/pull_request.csv')
pr_task_type_df = pd.read_csv('data/filtered/pr_task_type.csv')

## Manual Labelling Checking

In [2]:
# Load completed rater files and merge with original LLM labels
rater1_df = pd.read_csv('rq2_task_labels_rater1.csv').rename(columns={'task_type_label': 'rater1'})
rater2_df = pd.read_csv('rq2_task_labels_rater2.csv').rename(columns={'task_type_label': 'rater2'})

ratings_df = rater1_df.merge(rater2_df[['html_url', 'rater2']], on='html_url')

original_pr_labels = pr_task_type_df.merge(
    filtered_df[['html_url', 'id']],
    on='id'
)

ratings_df = ratings_df.merge(
    original_pr_labels[['html_url', 'type', 'confidence']],
    on='html_url'
).rename(columns={'type': 'llm_label'})

print(ratings_df.shape)
ratings_df.head()

(300, 7)


,html_url,title,body,rater1,rater2,llm_label,confidence
0,https://github.com/pdfme/pdfme/pull/718,refactor(generator): replace PDF comparison wi...,# Replace PDF comparison with jest-image-snaps...,test,test,refactor,10
1,https://github.com/bruin-data/bruin/pull/830,Fix ingestr version update in action,The `.github/workflows/update-ingestr.yml` wor...,ci,ci,fix,10
2,https://github.com/langfuse/langfuse-docs/pull...,Fix LangFuse casing,## Summary\n- fix LangFuse -> Langfuse in docs...,docs,docs,fix,10
3,https://github.com/langfuse/langfuse-docs/pull...,docs: Publish Langfuse June email as blog post,Publish the Langfuse June email update as a ne...,docs,docs,docs,10
4,https://github.com/giselles-ai/giselle/pull/909,feat: implement Query Node with RAG functional...,## Summary\n\nThis PR introduces the Query Nod...,feat,feat,feat,10


In [3]:
# --- Cohen's Kappa (rater1 vs rater2) ---
kappa_r1_r2 = cohen_kappa_score(ratings_df['rater1'], ratings_df['rater2'])
print(f"Cohen's Kappa (Rater 1 vs Rater 2): {kappa_r1_r2:.4f}")

# --- Raw percent agreement ---
pct_agree_raters = (ratings_df['rater1'] == ratings_df['rater2']).mean()
print(f"Raw agreement (Rater 1 vs Rater 2): {pct_agree_raters:.2%}")

Cohen's Kappa (Rater 1 vs Rater 2): 0.8104
Raw agreement (Rater 1 vs Rater 2): 84.67%


In [4]:
# --- Cohen's Kappa (each rater vs LLM) ---
kappa_r1_llm = cohen_kappa_score(ratings_df['rater1'], ratings_df['llm_label'])
kappa_r2_llm = cohen_kappa_score(ratings_df['rater2'], ratings_df['llm_label'])

print(f"Cohen's Kappa (Rater 1 vs LLM): {kappa_r1_llm:.4f}")
print(f"Cohen's Kappa (Rater 2 vs LLM): {kappa_r2_llm:.4f}")

# --- Raw percent agreement ---
pct_r1_llm = (ratings_df['rater1'] == ratings_df['llm_label']).mean()
pct_r2_llm = (ratings_df['rater2'] == ratings_df['llm_label']).mean()
print(f"Raw agreement (Rater 1 vs LLM): {pct_r1_llm:.2%}")
print(f"Raw agreement (Rater 2 vs LLM): {pct_r2_llm:.2%}")

# --- Three-way agreement (all three agree) ---
pct_all_agree = (
    (ratings_df['rater1'] == ratings_df['rater2']) & 
    (ratings_df['rater2'] == ratings_df['llm_label'])
).mean()
print(f"Three-way agreement (Rater 1 = Rater 2 = LLM): {pct_all_agree:.2%}")

Cohen's Kappa (Rater 1 vs LLM): 0.6786
Cohen's Kappa (Rater 2 vs LLM): 0.7445
Raw agreement (Rater 1 vs LLM): 74.33%
Raw agreement (Rater 2 vs LLM): 80.00%
Three-way agreement (Rater 1 = Rater 2 = LLM): 71.00%
